In [ ]:
# End-to-end voice assistant pipeline: Record Speech → Transcribe with Whisper → Generate Response with Gemini → Clean Text → Convert to Speech with Piper → Play Audio Output.

In [ ]:
# Import all libraries required for audio recording, speech recognition, LLM interaction, text processing, and speech synthesis.
import os
import re
import subprocess
import whisper
import google.generativeai as genai
import sounddevice as sd
import numpy as np

from dotenv import load_dotenv
from scipy.io.wavfile import read, write
from IPython.display import Audio

In [ ]:
# Configure Gemini API, initialize Whisper model, and define audio recording parameters.
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

gemini_model = genai.GenerativeModel(
    "models/gemini-2.5-flash"
)

SAMPLE_RATE = 16000
RECORD_SECONDS = 5

model = whisper.load_model("small")

In [3]:
# Record audio from the microphone and save it as input.wav.
audio = sd.rec(
    int(RECORD_SECONDS * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="int16"
)

sd.wait()

write("input.wav", SAMPLE_RATE, audio)

print("Saved: input.wav")

Saved: input.wav


In [ ]:
# Load the recorded audio, preprocess it, and convert the speech into text using the Whisper model.
sample_rate, audio = read("input.wav")

audio = audio.astype(np.float32)

if audio.ndim > 1:
    audio = audio.mean(axis=1)

audio /= max(abs(audio).max(), 1.0)

result = model.transcribe(
    audio,
    fp16=False,
    language="en"
)

transcript = result["text"]

print("Transcript:")
print(transcript)

Transcript:
 What is bubble sort can you explain in simple language?


In [ ]:
# Send the transcribed text to Gemini and generate a text-based response from the language model.
response = gemini_model.generate_content(
    f"""
    You are a voice assistant.

    Respond in plain text only.
    Do not use markdown.
    Do not use bullet points.
    Do not use asterisks.
    Do not use headings.

    User said:
    {transcript}
    """
)

assistant_response = response.text

In [ ]:
# Clean the generated response by removing markdown symbols and unwanted formatting before text-to-speech conversion.
assistant_response = re.sub(
    r'[*_`#>-]+',
    ' ',
    assistant_response
)

assistant_response = re.sub(
    r'\[(.*?)\]\((.*?)\)',
    r'\1',
    assistant_response
)

assistant_response = re.sub(
    r'\s+',
    ' ',
    assistant_response
).strip()

print("LLM Response:")
print(assistant_response)

LLM Response:
Bubble sort is a straightforward way to organize a list of items, like numbers, from smallest to largest or vice versa. Imagine you have a row of numbers that are mixed up. Bubble sort works by repeatedly going through this list, looking at pairs of numbers that are right next to each other. Here is how it works in simple terms: You start at the very beginning of your list. You compare the first number with the second number. If the first number is larger than the second, you swap their positions. Then, you move on to compare the second number (which might have just been swapped into that spot) with the third number. Again, if the second is larger, you swap them. You continue this process, comparing each adjacent pair all the way to the end of the list. After one complete pass through the list, the largest number will have "bubbled up" to its correct position at the very end of the list. It is like the heaviest bubble sinking to the bottom, or the largest number rising to

In [ ]:
# Save the cleaned language model response to a text file for logging and speech synthesis.
with open(
    "response.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(assistant_response)

print("Saved: response.txt")

Saved: response.txt


In [ ]:
# Convert the response text into speech using Piper and save the generated audio as reply.wav.
with open(
    "response.txt",
    "r",
    encoding="utf-8"
) as stdin:

    subprocess.run(
        [
            "piper/piper.exe",
            "--model",
            "piper/en_US-amy-medium.onnx",
            "--config",
            "piper/en_US-amy-medium.onnx.json",
            "--output_file",
            "reply.wav"
        ],
        stdin=stdin,
        check=True
    )

print("Saved: reply.wav")

Saved: reply.wav


In [ ]:
# Play the synthesized audio response generated by the voice assistant.
Audio("reply.wav")